# Telecom Customer Analytics Project
## Step 1: Import Libraries and Load Dataset

In [ ]:
import pandas as pd
field_Description=pd.read_excel("Field Descriptions.xlsx")
telecom_data = pd.read_excel("telcom_data (2).xlsx")
# to get the top 5 rows of the dataset
field_Description.head()
telecom_data.head()



## Step 2: Understanding the Dataset


In [ ]:
telecom_data.shape
telecom_data.info()
telecom_data.isnull().sum().sort_values(ascending=False)



## Step 3: Data Cleaning


In [ ]:
# checking the duplicates 
telecom_data.duplicated().sum()

### Interpretation

There are no duplicate records in the telecom dataset. Therefore, no duplicate rows need to be removed before further analysis.


### 3.1 Check Data Types

In [ ]:
telecom_data.dtypes

### Interpretation

The dataset contains three main data types:
- **float64**: Numerical variables such as download, upload, duration, and throughput.
- **object/string**: Categorical variables such as handset manufacturer and handset type.
- **datetime**: Date and time variables such as Start and End.

Understanding the data types helps us apply the appropriate data cleaning techniques to each type of column.

### 3.2 Create a Backup Copy of the Dataset

In [ ]:
telecom_clean = telecom_data.copy()


A copy of the original dataset is created before cleaning so that the original data remains unchanged. This allows us to return to the raw dataset if needed.

### 3.3 Identifying the numerical columns


In [ ]:
# Identifying the numerical columns

numeric_columns = telecom_clean.select_dtypes(include=['float64', 'int64']).columns

numeric_columns

### Interpretation

The numeric columns in the dataset have been identified successfully. These columns will be used to replace missing values with the mean, ensuring that only numerical features are modified while text and datetime columns remain unchanged.

###  3.4 Replace Missing Values with Mean

In [ ]:
# Replace Missing Values with Mean
telecom_clean[numeric_columns] = telecom_clean[numeric_columns].fillna(
    telecom_clean[numeric_columns].mean() 
)



### Verify that the missing values were filled 

In [ ]:
telecom_clean.isnull().sum().sort_values(ascending=False)


### Interpretation

The missing values in all numerical columns have been successfully replaced using the mean of their respective columns.

The remaining missing values belong to categorical (text) columns and datetime columns. These require different treatment because replacing text or dates with a numerical mean is not appropriate.

### Step 3.6: Replace Missing Values in Categorical Columns with Mode

In [ ]:
categorical_columns = telecom_clean.select_dtypes(include=['object']).columns

categorical_columns

### Fill Missing Values with Mode

In [ ]:
for column in categorical_columns:
    telecom_clean[column] = telecom_clean[column].fillna(
        telecom_clean[column].mode()[0]
    )

telecom_clean.isnull().sum().sort_values(ascending=False)

### 3.7 Handle Missing Datetime Values

In [ ]:
telecom_clean.dropna(subset=['Start', 'End'], inplace=True)
telecom_clean.isnull().sum().sort_values(ascending=False)


### Data Cleaning Summary

The telecom dataset was cleaned using the following steps:

- Checked for duplicate records (no duplicates found).
- Identified data types of all columns.
- Created a backup copy of the original dataset.
- Replaced missing values in numerical columns using the mean.
- Replaced missing values in categorical columns using the mode.
- Removed rows containing missing datetime values.
- Verified that no missing values remained after cleaning.

The dataset is now clean and ready for Exploratory Data Analysis (EDA).

## 4: Exploratory Data Analysis (EDA)

### 4.1 Summary Statistics

In [ ]:
telecom_clean.describe()

### Interpretation

The summary statistics provide an overview of the numerical variables in the telecom dataset.

- The dataset contains **150,000 cleaned observations**.
- The mean values represent the average customer usage for each numerical variable.
- The standard deviation indicates considerable variability in customer behavior.
- The minimum and maximum values show that some variables contain extremely large values, suggesting possible outliers.
- The quartiles (25%, 50%, and 75%) describe the distribution of the data and provide insight into the spread of customer usage.
- Overall, the dataset is ready for visualization and deeper exploratory analysis.

## 4.2 Distribution of Session Duration

A histogram is used to visualize the distribution of session duration. It helps us understand how frequently different session durations occur and whether the data is normally distributed or skewed.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.hist(telecom_clean['Dur. (ms)'], bins=30, edgecolor='black')

plt.title('Distribution of Session Duration')
plt.xlabel('Duration (ms)')
plt.ylabel('Frequency')

plt.show()

### Interpretation

The histogram shows the distribution of customer session duration.

- Most telecom sessions are concentrated within shorter durations.
- The distribution is positively (right) skewed, indicating that a small number of customers have significantly longer session durations.
- A few extreme values are visible, suggesting the presence of heavy users or outliers.
- The data is not normally distributed, which is common in telecom usage datasets.


## 4.3 Box Plot of Session Duration

A box plot is used to identify the spread of the data and detect potential outliers. It summarizes the distribution using the median, quartiles, and extreme values.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.boxplot(telecom_clean['Dur. (ms)'], orientation='horizontal') 

plt.title('Box Plot of Session Duration')
plt.xlabel('Duration (ms)')

plt.show()

### Interpretation

The box plot indicates that session duration contains a large number of outliers. Most customer sessions are concentrated within a relatively small range, while a few sessions have exceptionally long durations. This suggests that the distribution is positively skewed and that some users spend significantly more time on the network than the average customer. 

### 4.4: Histogram of Total Download (Bytes)

### Why are we plotting this?

A histogram of Total Download (Bytes) helps us understand how internet download usage is distributed among customers. It shows whether most customers download similar amounts of data or whether a few customers consume significantly higher amounts.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(telecom_clean['Total DL (Bytes)'],
         bins=30,
         edgecolor='black')

plt.title("Distribution of Total Download")
plt.xlabel("Total Download (Bytes)")
plt.ylabel("Frequency")

plt.show()

### Interpretation

The histogram shows that Total Download (Bytes) is fairly evenly distributed across the dataset. Unlike session duration, download usage does not exhibit a strong positive skew. Customers are spread across different download usage levels, indicating a wide variety of internet consumption patterns.

## 4.5 Box Plot of Total Download


A box plot summarizes the distribution of Total Download (Bytes) and helps identify potential outliers among customers with unusually high or low download usage.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.boxplot(
    telecom_clean['Total DL (Bytes)'],
    orientation='horizontal'
)

plt.title("Box Plot of Total Download")
plt.xlabel("Total Download (Bytes)")

plt.show()

### Interpretation

The box plot shows that Total Download (Bytes) is fairly evenly distributed with no significant outliers. The median is positioned near the center of the box, indicating a balanced distribution. The spread of the data suggests that customers have a wide range of download usage, but there are no unusually extreme download values.

### 4.4 Correlation Heatmap 


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14,10))

sns.heatmap(
    telecom_clean.select_dtypes(include=['float64','int64']).corr(),
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Heatmap")
plt.show()

### Interpretation

- The heatmap displays the correlation between all numerical variables.
- Dark red colors indicate a strong positive correlation.
- Dark blue colors indicate a strong negative correlation.
- Light colors indicate little or no correlation.
- Most variables show weak to moderate correlations, while a few download and upload-related variables have stronger positive relationships.
- The heatmap helps identify important features for further analysis and predictive modeling.

## 4.5 Scatter Plot: Total Download vs Total Upload

A scatter plot is used to visualize the relationship between two numerical variables. It helps identify positive or negative correlations, clusters, and unusual observations.

Here, we compare Total Download (Bytes) with Total Upload (Bytes) to understand whether customers with higher download usage also tend to have higher upload usage.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

sns.scatterplot(
    x=telecom_clean['Total DL (Bytes)'],
    y=telecom_clean['Total UL (Bytes)'],
    alpha=0.5
)

plt.title("Total Download vs Total Upload")
plt.xlabel("Total Download (Bytes)")
plt.ylabel("Total Upload (Bytes)")

plt.show()

### Interpretation:

The scatter plot compares Total Download (Bytes) and Total Upload (Bytes).
The data points are widely scattered across the graph.
There is no clear linear relationship between download and upload traffic.
This suggests that customers with higher download usage do not necessarily have higher upload usage.
Further statistical analysis (such as the correlation coefficient) would be needed to measure the strength of the relationship.

## 4.6 Top 5 Handsets per Top 3 Manufacturers

In [ ]:
# Find the top 3 handset manufacturers
top_manufacturers = telecom_clean['Handset Manufacturer'].value_counts().head(3)

top_manufacturers

## Top 10 Handset Manufacturers

In [ ]:
import matplotlib.pyplot as plt

top_manufacturers = telecom_clean['Handset Manufacturer'].value_counts().head(10)

plt.figure(figsize=(10,6))
top_manufacturers.plot(kind='bar')

plt.title("Top 10 Handset Manufacturers")
plt.xlabel("Manufacturer")
plt.ylabel("Number of Users")
plt.xticks(rotation=45)

plt.show()

### Interpretation:

Apple is the most commonly used handset manufacturer, with approximately 60,000 users.
Samsung is the second most popular manufacturer, followed by Huawei.
A significant number of records are labeled as 'undefined', which may indicate missing or unidentified manufacturer information.
Other manufacturers such as Sony, Xiaomi, OnePlus, Lenovo, and Wiko have comparatively fewer users.
This indicates that the telecom company's customer base is largely dominated by Apple, Samsung, and Huawei devices.

## Top 10 Handset Types

In [ ]:
top_handsets = telecom_clean['Handset Type'].value_counts().head(10)

plt.figure(figsize=(12,6))
top_handsets.plot(kind='bar')

plt.title("Top 10 Handset Types")
plt.xlabel("Handset Type")
plt.ylabel("Number of Users")
plt.xticks(rotation=45)

plt.show()

### Interpretation:

The graph displays the ten most frequently used handset models. A small number of handset models account for a large proportion of users, indicating that customer preferences are concentrated on a limited set of devices. This information can help the telecom company prioritize device-specific optimization, customer support, and marketing efforts.

The Huawei B528S-23A is the most widely used handset type, followed by several Apple iPhone models such as the iPhone 6S, iPhone 6, iPhone 7, iPhone SE, iPhone 8, and iPhone X. The presence of 'undefined' records indicates missing handset information that should be investigated to improve data quality. Overall, the telecom company can use these insights to optimize network compatibility, improve customer support, and design targeted marketing campaigns for the most popular devices.

# 5. Application Usage Analysis

This section analyzes the total network traffic generated by different applications such as Social Media, Google, Email, YouTube, Netflix, Gaming, and Other services. The objective is to identify which applications consume the highest amount of network resources.

In [ ]:
applications = {
    'Social Media': telecom_clean['Social Media DL (Bytes)'].sum() + telecom_clean['Social Media UL (Bytes)'].sum(),
    'Google': telecom_clean['Google DL (Bytes)'].sum() + telecom_clean['Google UL (Bytes)'].sum(),
    'Email': telecom_clean['Email DL (Bytes)'].sum() + telecom_clean['Email UL (Bytes)'].sum(),
    'Youtube': telecom_clean['Youtube DL (Bytes)'].sum() + telecom_clean['Youtube UL (Bytes)'].sum(),
    'Netflix': telecom_clean['Netflix DL (Bytes)'].sum() + telecom_clean['Netflix UL (Bytes)'].sum(),
    'Gaming': telecom_clean['Gaming DL (Bytes)'].sum() + telecom_clean['Gaming UL (Bytes)'].sum(),
    'Other': telecom_clean['Other DL (Bytes)'].sum() + telecom_clean['Other UL (Bytes)'].sum()
}

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.bar(
    applications.keys(),
    applications.values(),
    edgecolor='black'
)

plt.title("Application Usage by Total Traffic")
plt.xlabel("Applications")
plt.ylabel("Total Traffic (Bytes)")
plt.xticks(rotation=45)

plt.show()

### Interpretation:
The chart shows the total network traffic generated by different applications. Gaming and Other applications generate the highest traffic by a significant margin, followed by YouTube and Netflix. Google, Social Media, and Email contribute comparatively less network traffic. This indicates that entertainment and miscellaneous applications consume the majority of the telecom network bandwidth.

Business Insight:
The telecom company should prioritize network resources for Gaming and Other applications, as they account for the largest share of traffic. Understanding application usage patterns helps optimize bandwidth allocation, improve network performance, and enhance customer experience during peak usage periods.

Business Recommendation:
Increase network capacity for high-traffic applications such as Gaming and Other services. Monitor usage trends regularly and optimize network infrastructure to reduce congestion and improve service quality. Partnering with major content providers can further enhance network efficiency.

# 6. Executive Summary

This Telecom Customer Analytics Project analyzed customer network usage, handset information, and application traffic using Python. The dataset was cleaned by handling missing values and preparing numerical and categorical variables for analysis.

Exploratory Data Analysis (EDA) was performed using histograms, boxplots, correlation heatmaps, scatter plots, and bar charts to identify customer behavior and usage patterns.

The analysis provides insights into handset popularity, application traffic distribution, and network usage trends that can help improve business decision-making and network optimization.

# 7. Key Findings

• The dataset contained missing values that were successfully handled using mean and mode imputation.

• Most numerical variables showed skewed distributions with several outliers.

• The correlation heatmap indicated only a few strong relationships between network-related variables.

• There is no strong relationship between Total Download and Total Upload traffic.

• Apple, Samsung, and Huawei are the most widely used handset manufacturers.

• Huawei B528S-23A is the most commonly used handset model.

• Gaming and Other applications generate the highest amount of network traffic.

• These insights can help improve bandwidth allocation, customer service, and infrastructure planning.

# 8. Final Business Recommendations

• Increase network capacity for high-traffic applications such as Gaming and Other.

• Focus customer support and software optimization on the most commonly used handset manufacturers and models.

• Continuously monitor application traffic trends to improve Quality of Service (QoS).

• Optimize network resources during peak traffic hours to reduce congestion.

• Improve data collection processes to minimize missing or undefined handset information.

• Use customer usage patterns for future marketing campaigns and service planning.

# 9. Conclusion

The Telecom Customer Analytics Project successfully explored customer usage behavior, device popularity, and application traffic using Python.

The project demonstrates the complete data analysis workflow, including data cleaning, exploratory data analysis, visualization, and business insights.

The findings can support telecom companies in making data-driven decisions related to network optimization, customer experience, and strategic planning.
